In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 06. Search the archive

## What TF-IDF and cosine similarity mean, in plain words
TF-IDF turns a title into a list of numbers, one per word. A word scores higher if it is frequent in this title but rare across all titles. That way, the word the is almost ignored, while the word tariff stands out.

Cosine similarity compares two such lists and returns a score from 0 to 1. A score near 1 means the two texts use the same distinctive words in similar proportions. A score near 0 means they share almost nothing distinctive. It does not understand meaning the way a person does. It matches words.

## How to use the search below
Type a plain question as the query. The function returns the closest titles, with their EO number, date, President, and similarity score. Try your own queries after running the example cells. If a query returns odd results, that is useful information: it shows the limits of word matching.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
DF["search_text"] = DF["title_clean"].fillna("") + " " + DF["disposition_notes"].fillna("")
vec = TfidfVectorizer(stop_words="english", max_features=8000)
M = vec.fit_transform(DF["search_text"])
print("Search index built over", M.shape[0], "orders and", M.shape[1], "words.")
def search(query, top_n=5):
    q = vec.transform([query])
    scores = cosine_similarity(q, M).flatten()
    idx = scores.argsort()[::-1][:top_n]
    out = []
    for i in idx:
        r = DF.iloc[i]
        out.append({"score": round(float(scores[i]), 3), "eo": r["executive_order_number"], "date": str(r["date_signed"].date()), "president": r["president"], "title": r["title_clean"]})
    return pd.DataFrame(out)
print("Search function ready. Example queries follow.")

Search index built over 10879 orders and 8000 words.
Search function ready. Example queries follow.


In [4]:
for q in ["national parks and public lands", "civil rights and discrimination", "tariffs and trade"]:
    print("Query:", q)
    print(search(q, 5).to_string(index=False))
    print()

Query: national parks and public lands
 score      eo       date             president                                                                                                                                                                          title
 0.526  6136.0 1933-05-20 Franklin D. Roosevelt Extending the Application of the Act for the Relief of Unemployment Through the Performance of Useful Public Work to County Parks and Metropolitan Sections of Municipal Parks
 0.464  5680.0 1931-08-10        Herbert Hoover                   Amendment of Schedule A, to Include Superintendents or Officers in Charge of National Parks or Reservations, with Their Positions, in the Classified Service
 0.418 14314.0 2025-07-03       Donald J. Trump                                                                                                                 Making America Beautiful Again by Improving Our National Parks
 0.333  8009.0 1938-11-18 Franklin D. Roosevelt                      

In [5]:
for q in ["public health pandemic", "federal workforce pay", "immigration border"]:
    print("Query:", q)
    print(search(q, 5).to_string(index=False))
    print()

Query: public health pandemic
 score      eo       date            president                                                                                                                                                        title
 0.382  5700.0 1931-08-31       Herbert Hoover Permitting Public Health Service Officers to Serve in State, etc., Health Organizations, and Permitting Certain Appointments Under the Public Health Service
 0.358 14002.0 2021-01-22      Joseph R. Biden                                                                                                             Economic Relief Related to the COVID-19 Pandemic
 0.324 10506.0 1953-12-10 Dwight D. Eisenhower                                                                            Delegating Certain Functions of the President Under the Public Health Service Act
 0.321 13996.0 2021-01-21      Joseph R. Biden                Establishing the COVID-19 Pandemic Testing Board and Ensuring a Sustainable Public Health Wo

In [6]:
for q in ["artificial intelligence", "climate and clean energy"]:
    print("Query:", q)
    print(search(q, 5).to_string(index=False))
    print()

Query: artificial intelligence
 score      eo       date       president                                                              title
 0.606 14277.0 2025-04-23 Donald J. Trump     Advancing Artificial Intelligence Education for American Youth
 0.570 14409.0 2026-06-02 Donald J. Trump Promoting Advanced Artificial Intelligence Innovation and Security
 0.493 13859.0 2019-02-11 Donald J. Trump         Maintaining American Leadership in Artificial Intelligence
 0.477 14365.0 2025-12-11 Donald J. Trump   Ensuring a National Policy Framework for Artificial Intelligence
 0.476 14355.0 2025-09-30 Donald J. Trump  Unlocking Cures for Pediatric Cancer With Artificial Intelligence

Query: climate and clean energy
 score      eo       date     president                                                                                             title
 0.370 12038.0 1978-02-03  Jimmy Carter                                                                              Department of Energy
 0.370

## How this plugs into a real large language model system, without pretending it is one
No large language model was trained or used in this project. The search above is statistical: it matches words. A modern retrieval system for a language model, often called retrieval augmented generation, would do the following steps, in order.

1. Embeddings: replace word counts with dense vectors learned from large amounts of text, so that tariff and import duty can be close even though the words differ. TF-IDF cannot do that.
2. Vector store: store one embedding per order, or per chunk of an order, in a database built for similarity search.
3. Retrieval: for a user question, embed the question, find the nearest order chunks, and return them as evidence.
4. Generation: give those chunks to a language model with instructions to answer only from them, and to cite the EO number and date for each claim.

That pipeline is more powerful, and it has new risks: the model can paraphrase badly, invent a citation, or answer from its own training instead of the retrieved text. The TF-IDF search in this notebook is the honest baseline for that pipeline. If a future version adds embeddings, it should be evaluated against this baseline on the same queries, and every answer should still be checked against the order itself.

## Try it yourself
Change the query strings in the cells above, rerun, and compare. Good test queries: land withdrawal, federal pay, emergency board railroad dispute, veterans benefits. Notice which queries work well, because they use the exact words in titles, and which fail, because they use modern synonyms the old titles never used.